# 01 - Data Quality & Exploratory Data Analysis

**E-Commerce Revenue & Customer Analytics**

This notebook:
1. Loads the cleaned, feature-engineered datasets produced by `scripts/run_pipeline.py`
2. Profiles data quality (nulls, duplicates, ranges) as a sanity check
3. Explores revenue, profit, and order trends over time
4. Explores category and product performance
5. Explores customer, geographic, payment, and shipping patterns

> **Note on the data:** This project uses a **synthetic** e-commerce dataset,
> generated with a fixed random seed (`scripts/generate_data.py`). It is
> designed to look and behave like a real e-commerce export, including
> realistic data quality issues that get cleaned in `scripts/clean_data.py`.

Run `python scripts/run_pipeline.py` from the project root before executing this notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROCESSED_DIR = os.path.join("..", "data", "processed")

## 1. Load data

In [ ]:
customers = pd.read_csv(os.path.join(PROCESSED_DIR, "customers_clean.csv"), parse_dates=["signup_date"])
products = pd.read_csv(os.path.join(PROCESSED_DIR, "products_clean.csv"))
orders = pd.read_csv(os.path.join(PROCESSED_DIR, "orders_enriched.csv"), parse_dates=["order_date", "shipping_date", "delivery_date"])
order_items = pd.read_csv(os.path.join(PROCESSED_DIR, "order_items_enriched.csv"))
reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "reviews_clean.csv"), parse_dates=["review_date"])
shipping = pd.read_csv(os.path.join(PROCESSED_DIR, "shipping_clean.csv"), parse_dates=["shipping_date", "delivery_date"])

print(f"customers:   {customers.shape}")
print(f"products:    {products.shape}")
print(f"orders:      {orders.shape}")
print(f"order_items: {order_items.shape}")
print(f"reviews:     {reviews.shape}")
print(f"shipping:    {shipping.shape}")

## 2. Data quality profile

A quick sanity check now that the cleaning pipeline has already run -- this is the kind of profiling you'd normally do *before* cleaning, shown here to confirm the cleaned tables are actually clean.

In [ ]:
quality_report = pd.DataFrame({
    "table": ["customers", "products", "orders", "order_items", "reviews", "shipping"],
    "rows": [len(customers), len(products), len(orders), len(order_items), len(reviews), len(shipping)],
    "duplicate_pk": [
        customers["customer_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        order_items["order_item_id"].duplicated().sum(),
        reviews["review_id"].duplicated().sum(),
        shipping["shipping_id"].duplicated().sum(),
    ],
    "total_nulls": [
        customers.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        order_items.isna().sum().sum(),
        reviews.isna().sum().sum(),
        shipping.isna().sum().sum(),
    ],
})
quality_report

**What does this tell the business?** Zero duplicate primary keys confirms the cleaning pipeline correctly deduplicated every table. Remaining nulls (e.g. in `delivery_date` for orders that are still `Processing`/`Cancelled`) are expected and not a data quality defect.

In [ ]:
orders["order_status"].value_counts(normalize=True).mul(100).round(2)

## 3. Revenue & profit trend over time

In [ ]:
valid_orders = orders[orders["order_status"] != "Cancelled"].copy()

monthly = valid_orders.groupby("year_month").agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    orders=("order_id", "count"),
).reset_index().sort_values("year_month")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["year_month"], monthly["revenue"], marker="o", label="Revenue", linewidth=2)
ax.plot(monthly["year_month"], monthly["profit"], marker="o", label="Profit", linewidth=2)
ax.set_title("Monthly Revenue & Profit Trend", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Amount (Rs)")
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

**What does this tell the business?** Revenue and profit move together month over month, confirming margins are relatively stable rather than being propped up in a handful of months. Any month where the profit line diverges sharply downward from revenue is worth investigating for discounting or cost issues.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly["year_month"], monthly["orders"], color="#4C72B0")
ax.set_title("Monthly Order Volume", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Orders")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 4. Category & product performance

In [ ]:
item_orders = order_items.merge(valid_orders[["order_id"]], on="order_id", how="inner")

category_revenue = item_orders.groupby("category").agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).reset_index().sort_values("revenue", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(category_revenue["category"], category_revenue["revenue"], color="#55A868")
axes[0].set_title("Revenue by Category", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Revenue (Rs)")
axes[0].invert_yaxis()

axes[1].barh(category_revenue["category"], category_revenue["profit"], color="#C44E52")
axes[1].set_title("Profit by Category", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Profit (Rs)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

**What does this tell the business?** Electronics and Fashion dominate revenue given their higher price points, but a category that ranks highly on revenue and poorly on profit is a signal that its pricing or discount strategy needs review -- see the discount-vs-profit analysis in notebook 04.

In [ ]:
top_products = item_orders.groupby(["product_id"]).agg(
    revenue=("revenue", "sum")
).reset_index().merge(products[["product_id", "product_name", "category"]], on="product_id").sort_values("revenue", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_products["product_name"], top_products["revenue"], color="#8172B2")
ax.set_title("Top 10 Products by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue (Rs)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Geographic distribution

In [ ]:
geo_revenue = valid_orders.groupby("state").agg(
    revenue=("revenue", "sum"),
    customers=("customer_id", "nunique"),
).reset_index().sort_values("revenue", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(geo_revenue["state"], geo_revenue["revenue"], color="#64B5CD")
ax.set_title("Top 15 States by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue (Rs)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**What does this tell the business?** Revenue is concentrated in a handful of states -- typically the most populous / most economically active ones. This supports targeted regional marketing spend and can guide warehouse/fulfillment center placement decisions.

## 6. Payment method mix

In [ ]:
payment_mix = orders["payment_method"].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(payment_mix.values, labels=payment_mix.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("Set2", len(payment_mix)))
ax.set_title("Payment Method Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Shipping performance

In [ ]:
ship_delivery = shipping.merge(orders[["order_id", "delivery_days"]], on="order_id", how="left")

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=ship_delivery, x="shipping_method", y="delivery_days", hue="shipping_method", ax=ax, palette="pastel", legend=False)
ax.set_title("Delivery Days by Shipping Method", fontsize=14, fontweight="bold")
ax.set_xlabel("Shipping Method")
ax.set_ylabel("Delivery Days")
plt.tight_layout()
plt.show()

**What does this tell the business?** Same-Day and Express shipping should show a tight, low distribution of delivery days. If Standard/Economy show a long right tail (many outliers taking far longer than typical), that's an operational red flag worth investigating with the logistics team -- and it's very likely dragging down review ratings (see notebook 04).

## 8. Review ratings

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
rating_counts = reviews["rating"].value_counts().sort_index()
ax.bar(rating_counts.index.astype(str), rating_counts.values, color="#DD8452")
ax.set_title("Review Rating Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Rating (1-5 stars)")
ax.set_ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

print(f"Average rating: {reviews['rating'].mean():.2f} / 5")

**What does this tell the business?** A rating distribution skewed toward 4-5 stars is typical and healthy. The 1-2 star tail, even if small in absolute terms, is worth cross-referencing against delivery time and category -- see the deeper dive in notebook 04 (`04_business_insights.ipynb`).

## Summary

This notebook confirmed the cleaned data is structurally sound (no duplicate keys, referential integrity holds) and surfaced the headline patterns: revenue/profit trend, category and product leaders, geographic concentration, payment mix, shipping performance, and review sentiment. The next notebook (`02_customer_analysis.ipynb`) goes deeper on customer-level behavior.